In [1]:
import kagglehub

path = kagglehub.dataset_download("viacheslavshalamov/russian-road-signs-segmentation-dataset")


/Users/anovichkovy/PyCharmMiscProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 401M/401M [01:06<00:00, 6.28MB/s] 

Extracting files...


In [3]:
# Переводим датасет из coco формата в yolo формат

import json
from pathlib import Path
from tqdm import tqdm


dataset_root = Path("../sign_dataset")
output_root = Path("yolo_dataset_1")
splits = ["train", "val"]

for split in splits:
    img_dir = dataset_root / split
    json_path = img_dir / "via_region_data.json"
    out_img_dir = output_root / "images" / split
    out_lbl_dir = output_root / "labels" / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    with open(json_path) as f:
        data = json.load(f)

    for item in tqdm(data.values(), desc=f"Processing {split}"):
        filename = item["filename"]
        width = item["file_attributes"]["width"]
        height = item["file_attributes"]["height"]
        regions = item["regions"]
        label_file = out_lbl_dir / (Path(filename).stem + ".txt")

        with open(label_file, "w") as f:
            for r in regions.values():
                shape = r["shape_attributes"]["name"]
                attr = r["shape_attributes"]
                if "all_points_x" not in attr or "all_points_y" not in attr:
                    continue
                xs = r["shape_attributes"]["all_points_x"]
                ys = r["shape_attributes"]["all_points_y"]
                if len(xs) < 3:
                    continue
                points = []
                for x, y in zip(xs, ys):
                    xn = x / width
                    yn = y / height
                    points.append(f"{xn:.6f}")
                    points.append(f"{yn:.6f}")
                line = "0 " + " ".join(points)
                f.write(line + "\n")

        src = img_dir / filename
        dst = out_img_dir / filename
        dst.write_bytes(src.read_bytes())


Processing val: 100%|██████████| 127/127 [00:00<00:00, 1689.24it/s]

In [4]:
data_yaml = """
path: yolo_dataset_1
train: images/train
val: images/val

nc: 1
names: ["road_sign"]
"""

with open("dataset-3_1.yaml", "w") as f:
    f.write(data_yaml)

In [9]:
from ultralytics import YOLO

model = YOLO("../yolo11n-seg.pt")

model.train(
    data="dataset-3_1.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device="mps",
    workers=12
)

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.14.3 torch-2.10.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset-3_1.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, 

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x12de8b150>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    

In [13]:
from ultralytics import YOLO
import cv2

model = YOLO("runs/segment/train3/weights/best.pt")

video_path = "video3.mp4"
cap = cv2.VideoCapture(video_path)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    "video3_result.mp4",
    fourcc,
    cap.get(cv2.CAP_PROP_FPS),
    (
        int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    ),
)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame)
    annotated = results[0].plot()
    out.write(annotated)

cap.release()
out.release()


0: 640x384 (no detections), 42.7ms
Speed: 1.3ms preprocess, 42.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 30.4ms
Speed: 1.2ms preprocess, 30.4ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 27.7ms
Speed: 0.7ms preprocess, 27.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 30.9ms
Speed: 0.8ms preprocess, 30.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 44.7ms
Speed: 2.3ms preprocess, 44.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 34.4ms
Speed: 0.7ms preprocess, 34.4ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 42.0ms
Speed: 0.8ms preprocess, 42.0ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 29.0ms
Speed: 1.0ms preprocess, 29.0ms i

In [17]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm

model_path = "runs/segment/train3/weights/best.pt"

images_dir = Path("yolo_dataset_1/images/val")
labels_dir = Path("yolo_dataset_1/labels/val")

model = YOLO(model_path)

ious = []
precisions = []
recalls = []
l2s = []

def load_gt_mask(label_path, shape):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    if not label_path.exists():
        return mask
    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        coords = list(map(float, parts[1:]))
        pts = np.array(coords).reshape(-1,2)
        pts[:,0] *= w
        pts[:,1] *= h
        pts = pts.astype(np.int32)
        cv2.fillPoly(mask, [pts], 1)
    return mask


for img_path in tqdm(list(images_dir.glob("*.jpg"))):
    label_path = labels_dir / (img_path.stem + ".txt")

    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    gt_mask = load_gt_mask(label_path, (h, w))
    result = model(img)[0]
    pred_mask = np.zeros((h,w), dtype=np.uint8)
    if result.masks is not None:
        for m in result.masks.data.cpu().numpy():
            m = cv2.resize(m, (w,h))
            m = (m > 0.5).astype(np.uint8)
            pred_mask = np.logical_or(pred_mask, m)

    pred_mask = pred_mask.astype(np.uint8)
    TP = np.logical_and(gt_mask==1, pred_mask==1).sum()
    FP = np.logical_and(gt_mask==0, pred_mask==1).sum()
    FN = np.logical_and(gt_mask==1, pred_mask==0).sum()
    union = TP + FP + FN
    if union == 0:
        continue

    iou = TP / union
    precision = TP / (TP + FP) if (TP+FP) > 0 else 0
    recall = TP / (TP + FN) if (TP+FN) > 0 else 0
    l2 = np.sqrt(((gt_mask - pred_mask) ** 2).sum())
    ious.append(iou)
    precisions.append(precision)
    recalls.append(recall)
    l2s.append(l2)

ious = np.array(ious)

print("Mean IoU:", ious.mean())
print("Precision:", np.mean(precisions))
print("Recall:", np.mean(recalls))
print("L2:", np.mean(l2s))

print("IoU >=0.5:", np.mean(ious >= 0.5))
print("IoU >=0.75:", np.mean(ious >= 0.75))
print("IoU >=0.9:", np.mean(ious >= 0.9))

  0%|          | 0/127 [00:00<?, ?it/s]


0: 384x640 (no detections), 34.7ms
Speed: 1.6ms preprocess, 34.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 road_sign, 22.2ms
Speed: 0.7ms preprocess, 22.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 road_sign, 37.0ms
Speed: 0.7ms preprocess, 37.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)


  2%|▏         | 3/127 [00:00<00:05, 23.55it/s]


0: 384x640 1 road_sign, 121.0ms
Speed: 6.4ms preprocess, 121.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 640x480 1 road_sign, 45.7ms
Speed: 6.6ms preprocess, 45.7ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 480)

0: 384x640 2 road_signs, 20.2ms
Speed: 0.7ms preprocess, 20.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)


  5%|▍         | 6/127 [00:00<00:13,  9.14it/s]

...


0: 640x480 3 road_signs, 23.5ms
Speed: 1.3ms preprocess, 23.5ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 480)

0: 384x640 2 road_signs, 20.3ms
Speed: 0.7ms preprocess, 20.3ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 road_signs, 21.4ms
Speed: 0.7ms preprocess, 21.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)


 99%|█████████▉| 126/127 [00:04<00:00, 23.06it/s]


0: 384x640 1 road_sign, 19.6ms
Speed: 0.7ms preprocess, 19.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)


100%|██████████| 127/127 [00:04<00:00, 25.65it/s]

Mean IoU: 0.5022794457417861
Precision: 0.5983469617053474
Recall: 0.7380883021442112
L2: 82.4622358429711
IoU >=0.5: 0.5847457627118644
IoU >=0.75: 0.0847457627118644
IoU >=0.9: 0.00847457627118644


In [18]:
from ultralytics import YOLO
model = YOLO("runs/segment/train3/weights/best.pt")

video_path = "video1.mp4"

results = model.track(
    source=video_path,
    tracker="botsort.yaml",
    save=True,
    conf=0.3
)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/1.5 MB ? eta -:--:--
   ━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/1.5 MB 2.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━ 0.8/1.5 MB 1.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━ 1.0/1.5 MB 1.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━ 1.0/1.5 MB 1.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 1.3/1.5 MB 1.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 994.5 kB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

requirements: AutoUpdate success ✅ 4.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True`

In [19]:
from ultralytics import YOLO
model = YOLO("runs/segment/train3/weights/best.pt")

video_path = "video1.mp4"

results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    save=True,
    conf=0.3
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 45.6ms
video 1/1 (frame 2/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 30.3ms
video 1/1 (frame 3/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 26.4ms
video 1/1 (frame 4/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 27.7ms
video 1/1 (frame 5/903) /Users/anovichkovy/PyC

In [21]:
from ultralytics import YOLO
from collections import defaultdict

def count_id_switches(video_path, model_path, tracker="bytetrack.yaml", dist_threshold=50):
    model = YOLO(model_path)
    track_history = defaultdict(list)
    id_switches = 0
    results = model.track(
        source=video_path,
        tracker=tracker,
        stream=True,
        conf=0.3
    )
    for r in results:
        if r.boxes.id is None:
            continue
        ids = r.boxes.id.cpu().numpy()
        boxes = r.boxes.xyxy.cpu().numpy()
        for obj_id, box in zip(ids, boxes):
            center_x = (box[0] + box[2]) / 2
            center_y = (box[1] + box[3]) / 2
            track_history[obj_id].append((center_x, center_y))
            if len(track_history[obj_id]) > 1:
                prev = track_history[obj_id][-2]
                curr = track_history[obj_id][-1]
                dist = ((prev[0]-curr[0])**2 + (prev[1]-curr[1])**2)**0.5
                if dist > dist_threshold:
                    id_switches += 1

    return id_switches


video_path = "video1.mp4"
model_path = "runs/segment/train3/weights/best.pt"

bytetrack_switches = count_id_switches(video_path, model_path, tracker="bytetrack.yaml")
botsort_switches = count_id_switches(video_path, model_path, tracker="botsort.yaml")

print("ByteTrack ID Switches:", bytetrack_switches)
print("BoT-SORT ID Switches:", botsort_switches)


video 1/1 (frame 1/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 79.7ms
video 1/1 (frame 2/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 113.6ms
video 1/1 (frame 3/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 29.1ms
video 1/1 (frame 4/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 22.4ms
video 1/1 (frame 5/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 22.0ms
video 1/1 (frame 6/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 22.2ms
video 1/1 (frame 7/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 21.7ms
video 1/1 (frame 8/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 21.2ms
video 1/1 (frame 9/903) /Users/anovichkovy/PyCharmMiscProject/video1.mp4: 640x384 2 road_signs, 21.1ms
video 1/1 (frame 10/903) /Users/anovichkovy/PyCharmMiscProject/video1.m

In [22]:
from ultralytics import YOLO
from collections import defaultdict

def count_id_switches(video_path, model_path, tracker="bytetrack.yaml", dist_threshold=50):
    model = YOLO(model_path)
    track_history = defaultdict(list)
    id_switches = 0
    results = model.track(
        source=video_path,
        tracker=tracker,
        stream=True,
        conf=0.3
    )
    for r in results:
        if r.boxes.id is None:
            continue
        ids = r.boxes.id.cpu().numpy()
        boxes = r.boxes.xyxy.cpu().numpy()
        for obj_id, box in zip(ids, boxes):
            center_x = (box[0] + box[2]) / 2
            center_y = (box[1] + box[3]) / 2
            track_history[obj_id].append((center_x, center_y))
            if len(track_history[obj_id]) > 1:
                prev = track_history[obj_id][-2]
                curr = track_history[obj_id][-1]
                dist = ((prev[0]-curr[0])**2 + (prev[1]-curr[1])**2)**0.5
                if dist > dist_threshold:
                    id_switches += 1

    return id_switches


video_path = "video2.mp4"
model_path = "runs/segment/train3/weights/best.pt"

bytetrack_switches = count_id_switches(video_path, model_path, tracker="bytetrack.yaml")
botsort_switches = count_id_switches(video_path, model_path, tracker="botsort.yaml")

print("ByteTrack ID Switches:", bytetrack_switches)
print("BoT-SORT ID Switches:", botsort_switches)


video 1/1 (frame 1/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 4 road_signs, 25.0ms
video 1/1 (frame 2/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 38.1ms
video 1/1 (frame 3/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 43.7ms
video 1/1 (frame 4/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 53.5ms
video 1/1 (frame 5/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 40.8ms
video 1/1 (frame 6/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 47.0ms
video 1/1 (frame 7/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 63.4ms
video 1/1 (frame 8/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 64.0ms
video 1/1 (frame 9/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp4: 640x384 3 road_signs, 28.1ms
video 1/1 (frame 10/946) /Users/anovichkovy/PyCharmMiscProject/video2.mp

In [23]:
from ultralytics import YOLO
from collections import defaultdict

def count_id_switches(video_path, model_path, tracker="bytetrack.yaml", dist_threshold=50):
    model = YOLO(model_path)
    track_history = defaultdict(list)
    id_switches = 0
    results = model.track(
        source=video_path,
        tracker=tracker,
        stream=True,
        conf=0.3
    )
    for r in results:
        if r.boxes.id is None:
            continue
        ids = r.boxes.id.cpu().numpy()
        boxes = r.boxes.xyxy.cpu().numpy()
        for obj_id, box in zip(ids, boxes):
            center_x = (box[0] + box[2]) / 2
            center_y = (box[1] + box[3]) / 2
            track_history[obj_id].append((center_x, center_y))
            if len(track_history[obj_id]) > 1:
                prev = track_history[obj_id][-2]
                curr = track_history[obj_id][-1]
                dist = ((prev[0]-curr[0])**2 + (prev[1]-curr[1])**2)**0.5
                if dist > dist_threshold:
                    id_switches += 1

    return id_switches


video_path = "video3.mp4"
model_path = "runs/segment/train3/weights/best.pt"

bytetrack_switches = count_id_switches(video_path, model_path, tracker="bytetrack.yaml")
botsort_switches = count_id_switches(video_path, model_path, tracker="botsort.yaml")

print("ByteTrack ID Switches:", bytetrack_switches)
print("BoT-SORT ID Switches:", botsort_switches)


video 1/1 (frame 1/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 21.2ms
video 1/1 (frame 2/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 21.0ms
video 1/1 (frame 3/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 20.3ms
video 1/1 (frame 4/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 20.4ms
video 1/1 (frame 5/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 20.0ms
video 1/1 (frame 6/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 20.0ms
video 1/1 (frame 7/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 22.4ms
video 1/1 (frame 8/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 20.3ms
video 1/1 (frame 9/1013) /Users/anovichkovy/PyCharmMiscProject/video3.mp4: 640x384 (no detections), 20.1ms
video 1/1 (frame 10/1013) /Users/ano